<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day22_practice2_%EB%8F%84%EA%B5%AC%EC%99%80_%EB%A3%A8%ED%94%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 도구 3개 + Agent 루프
# 도구 3개 구현: 규정 검색 / 신고 조회 / 상태 조회
# Agent 루프 완성: 결정 → 실행 → 재질의 → 답(안전장치 포함)

In [17]:
import os, requests, json, re
import torch
import torch.nn.functional as F

def _get_api_key():
  try:
    from google.colab import userdata
    return userdata.get("VISION_API_KEY")
  except Exception:
    return os.environ.get("VISION_API_KEY", "") # Colab이 아니거나 Secret이 없을 때 환경변수 딕셔너리에서 꺼냄

API_KEY = _get_api_key()
# API 호출 - openAI 호환 api 형식으로 코드 작성
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai" # Gemini OpenAI 호환
MODEL = "gemini-3.5-flash-lite"

In [18]:
# 셀 1. 신고DB
REPORTS_DB = [
    {"id": 12, "hazard_type": "포트홀", "severity": 4, "location": "강남대로 사거리", "status": "처리중"},
    {"id": 15, "hazard_type": "포트홀", "severity": 4, "location": "서초대로 12",     "status": "접수"},
    {"id": 18, "hazard_type": "포트홀", "severity": 3, "location": "충무로 30",     "status": "완료"},
    {"id": 21, "hazard_type": "균열",   "severity": 2, "location": "을지로 골목",   "status": "접수"},
    {"id": 25, "hazard_type": "맨홀",   "severity": 3, "location": "안국역 3번출구", "status": "처리중"},
]
KNOWLEDGE = [
    "심각도 4 이상의 포트홀은 접수 후 24시간 이내에 긴급 보수팀을 배정한다.",
    "심각도 3 이하의 도로 균열은 주간 정기 점검 일정에 포함하여 7일 이내 처리한다.",
    "맨홀 뚜껑 이상 신고는 안전 펜스를 우선 설치하고 상수도사업소에 이관한다.",
    "처리 완료된 신고는 사진과 함께 신고자에게 앱 알림으로 통보한다.",
]

In [25]:
# 셀 2. 도구 3개 (파이썬 함수)
def _embed(text, vocab, n=2): # 문장들 2글자 단어 조각 벡터화
  v = torch.zeros(len(vocab))
  t = re.sub(r"\s+", "", text)
  for i in range(len(t) - n + 1):
    if t[i:i+n] in vocab:
      v[vocab[t[i:i+n]]] += 1
  return v

_grams = set()
for d in KNOWLEDGE:
  t = re.sub(r"\s+", "", d)
  _grams |= {t[i:i+2] for i in range(len(t) - 1)}
_vocab = {g: i for i, g in enumerate(sorted(_grams))} # 단어 사전 (단어조각: 번호)
_doc_vecs = torch.stack([_embed(d, _vocab) for d in KNOWLEDGE]) # 문장 벡터화

def search_regulation(query: str) -> str: # 도구1 : 규정 검색
  sims = F.cosine_similarity(_embed(query, _vocab).unsqueeze(0). _doc_vecs)
  return KNOWLEDGE[int(sims.argmax())]

def query_reports(hazard_type=None, min_severity=None) -> dict: # 도구2 : 신고 조회
  rows = [r for r in REPORTS_DB
            if (hazard_type is None or r["hazard_type"] == hazard_type)
            and (min_severity is None or r["severity"] >= min_severity)]
  return {"count": len(rows), "items": [f"#{r['id']} {r['location']} (심각도{r['severity']}, {r['status']})" for r in rows]}

def get_report_status(report_id: int) -> str: # 도구3 : 상태 조회
  for r in REPORTS_DB:
    if r["id"] == int(report_id):
      return f"#{r['id']} {r['hazard_type']} ({r['location']}) - 현재 '{r['status']}'"
  return f"신고번호 {report_id}를 찾을 수 없습니다"

# 도구 이름: TOOLS["이름"](**args) 사용
TOOLS = {"search_regulation": search_regulation,
         "query_reports": query_reports,
         "get_report_status": get_report_status}
print("도구 3개 등록:", list(TOOLS.keys()))
print("단독 테스트:", get_report_status(12))

도구 3개 등록: ['search_regulation', 'query_reports', 'get_report_status']
단독 테스트: #12 포트홀 (강남대로 사거리) - 현재 '처리중'


In [26]:
# 셀 3. '결정'부분 : real은 LLM
def decide_real(conversation):
  if not API_KEY:
    return None # decide_mock()이 실행됨
  try:
    r = requests.post(f"{GEMINI_BASE_URL}/chat/completions",
                      headers = {"Authorization": f"Bearer {API_KEY}"},
                      json = {"model": MODEL, "messages":conversation, "temperature":0},
                      timeout=30)
    r.raise_for_status() # HTTP 에러면 except로
    text = r.json()["choices"][0]["message"]["content"] # 응답 JSON에서 내용 가져오기
    text = re.sub(r"```(?:json)?", "", text) # 내용 중 코드펜스 제거
    return json.loads(text[text.find("{"):text.rfind("}")+1]) # 첫 {}만 잘라서 JSON 파싱
  except Exception as e:
    print(f"Gemini 호출/파싱 실패 → MOCK으로 대체: {type(e).__name__}: {e}")
    return None

def decide_mock(conversation):
  pass

def decide(conversation):
  return decide_real(conversation) or decide_mock(conversation)

In [27]:
# 셀 4. Agent 루프
# 흐름: 질문 → 판단(decide) →  도구 실행 → 결과를 대화에 추가 → 다시 판단 → answer 나오면 종료
SYSTEM = """너는 도로 신고 시스템 상담 Agent다. 도구:
1. search_regulation(query) 2. query_reports(hazard_type, min_severity) 3. get_report_status(report_id)
JSON 만 응답: {"tool":..., "args":{...}} 또는 {"answer":"..."}"""
MAX_TURNS = 4 # 최대 반복 횟수(무한루프 방지)

def run_agent(question, verbose = True):
  conv = [{"role": "system", "content": SYSTEM}, # 대화 기록
          {"role": "user", "content": question}]
  used_tools = []
  for turn in range(MAX_TURNS): # 안전 장치 1: 무한 반복 못하도록
    decision = decide(conv) # 판단: 무엇을 할지 고르는 일 → 이번 턴의 결정(tool 또는 answer)
    if "answer" in decision:
      return{"answer": decision["answer"], "used_tools": used_tools}

    tool_name = decision.get("tool", "") # LLM이 고른 도구 이름
    if tool_name not in TOOLS: # 안전 장치 2: 등록 안 된 도구면 실행 거부
      return{"answer": "요청을 처리할 수 없습니다.", "used_tools": used_tools}
    try:
      result = TOOLS[tool_name](**decision.get("args",{})) # 함수 코드가 실행, dict를 함수 인자로 펼침
    except Exception as e:
      result = f"도구 실행 오류: {e}" # 안전 장치 3: 오류를 '결과'로 만들어 LLM이 고치게 한다

    used_tools.append(tool_name) # 쓴 도구 기록
    if verbose: # 중간 과정 출력
      print(f"[턴{turn+1}] {tool_name}{decision.get('args')} → {str(result)[:56]}")
    conv.append({"role": "assistant", "content": json.dumps(decision, ensure_ascii= False)}) # 결정을 대화에 기록
    conv.append({"role": "user", "content": f"[도구 결과] {result}"}) # 결과를 다음 입력으로 되넣기
    # 매번 턴이 바뀔때마다 conv가 계속 추가된다
  return{"answer": "여러 번 시도했지만 답을 완성하지 못했습니다.", "used_tools": used_tools} # 턴을 다 써도 answer가 안 나오면

In [28]:
# 셀 5. 실행 - 질문 유형별로 다른 도구를 집는다
print(f"모드: {'REAL (Gemini)' if API_KEY else 'MOCK 라우터'}\n")
for q in ["안녕하세요",
          "포트홀 신고 몇 건이나 있어요?",
          "심각도 4 포트홀은 언제 처리되나요? 규정 알려줘",
          "12번 신고 상태 어떻게 돼요?",
          "안녕하세요"]:
  print(f"Q: {q}")
  out_agent = run_agent(q)
  print(f"A: {out_agent['answer'][:70]}")
  print(f"사용 도구: {out_agent['used_tools'] or '(없음)'}\n")

모드: REAL (Gemini)

Q: 안녕하세요
A: 안녕하세요! 도로 신고 시스템 상담 Agent입니다. 도로 파손, 가로등 고장 등 도로 관련 불편 사항을 신고하거나 처리 현황
사용 도구: (없음)

Q: 포트홀 신고 몇 건이나 있어요?
[턴1] query_reports{'hazard_type': '포트홀'} → {'count': 3, 'items': ['#12 강남대로 사거리 (심각도4, 처리중)', '#15 
A: 현재 포트홀 관련 신고는 총 3건이 있습니다.
1. #12 강남대로 사거리 (심각도 4, 처리중)
2. #15 서초대로 12 
사용 도구: ['query_reports']

Q: 심각도 4 포트홀은 언제 처리되나요? 규정 알려줘
[턴1] search_regulation{'query': '심각도 4 포트홀 처리 기한'} → 도구 실행 오류: 'Tensor' object has no attribute '_doc_vecs'
[턴2] search_regulation{'query': '포트홀 처리 규정'} → 도구 실행 오류: 'Tensor' object has no attribute '_doc_vecs'
A: 죄송합니다. 현재 내부 시스템 오류로 인해 규정 정보를 조회하는 데 문제가 발생했습니다. 포트홀 처리 기한에 대한 일반적인 규
사용 도구: ['search_regulation', 'search_regulation']

Q: 12번 신고 상태 어떻게 돼요?
[턴1] get_report_status{'report_id': '12'} → #12 포트홀 (강남대로 사거리) - 현재 '처리중'
A: 12번 신고는 현재 '처리중' 상태입니다.
사용 도구: ['get_report_status']

Q: 안녕하세요
A: 안녕하세요! 도로 신고 시스템 상담 Agent입니다. 도로 파손, 가로등 고장, 불법 주정차 등 도로 관련 불편 사항을 신고하
사용 도구: (없음)

